---
title: Downloading and reading MNIST dataset
format: 
  html:
    code-fold: false
---

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aavelozb/aavelozb.github.io/blob/main/ejemplos-ml/naive_Bayes-gc.ipynb)

### Loading the dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
!curl -O https://raw.githubusercontent.com/aavelozb/aavelozb.github.io/main/data/mnist.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8744k  100 8744k    0     0  13.3M      0 --:--:-- --:--:-- --:--:-- 13.3M


,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
5,"Subject: great nnews hello , welcome to medzo...",1
6,Subject: here ' s a hot play in motion homela...,1
7,Subject: save your money buy getting this thin...,1
8,Subject: undeliverable : home based business f...,1
9,Subject: save your money buy getting this thin...,1


In [ ]:

def load_idx_images(path):
    with open(path,'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data[16:].reshape(-1, 28*28) / 255.0

def load_idx_labels(path):
    with open(path,'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data[8:]

X_full = load_idx_images('mnist/train-images.idx3-ubyte')
y_full = load_idx_labels('mnist/train-labels.idx1-ubyte')
X_test = load_idx_images('mnist/t10k-images.idx3-ubyte')
y_test = load_idx_labels('mnist/t10k-labels.idx1-ubyte')

X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=10000, random_state=42)


im = X_train[10000, :].reshape(28,28)
print(y_train[10000])
plt.imshow(im)
plt.show()

In [5]:
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[made, corporate, ,, its, business, hotat, rea..."
1,Subject: the stock trading gunslinger fanny i...,1,"[perspicuous, attainder, is, clothesman, deoxy..."
2,Subject: unbelievable new homes made easy im ...,1,"[new, your, you, foward, made, ,, way, is, adv..."
3,Subject: 4 color printing special request add...,1,"[com, request, goldengraphix, ,, is, special, ..."
4,"Subject: do not have money , get software cds ...",1,"[?, old, t, be, ,, death, is, get, compatibili..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[shakedown, ,, va, blister, sh, ourselves, wel..."
6,Subject: here ' s a hot play in motion homela...,1,"[whiie, assumptions, made, security, historica..."
7,Subject: save your money buy getting this thin...,1,"[your, you, buy, ?, be, man, cialis, is, get, ..."
8,Subject: undeliverable : home based business f...,1,"[70590202251232, ,, reach, business, d, 13, kp..."
9,Subject: save your money buy getting this thin...,1,"[your, you, buy, ?, be, man, cialis, is, aicoh..."


In [6]:
num_emails = len(emails)
num_spam = sum(emails['spam'])

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)
print()

# Calculating the prior probability that an email is spam
print("Probability of spam:", num_spam/num_emails)

Number of emails: 5728
Number of spam emails: 1368

Probability of spam: 0.2388268156424581


### Training a naive Bayes model

Our plan is to write a dictionary, and in this dictionary record every word, and its pair of occurrences in spam and ham

In [7]:
model = {}

# Training process
for index, email in emails.iterrows():
    for word in email['words']:
        if word not in model:
            model[word] = {'spam': 1, 'ham': 1}
        if word in model:
            if email['spam']:
                model[word]['spam'] += 1
            else:
                model[word]['ham'] += 1

In [8]:
model['lottery']

{'spam': 9, 'ham': 1}

In [9]:
model['sale']

{'spam': 39, 'ham': 42}

### Using the model to make predictions

In [10]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]['spam']
    num_ham_with_word = model[word]['ham']
    return 1.0*num_spam_with_word/(num_spam_with_word + num_ham_with_word)

In [11]:
predict_bayes('lottery')

0.9

In [12]:
predict_bayes('sale')

0.48148148148148145

In [13]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails['spam'])
    num_ham = total - num_spam
    email = email.lower()
    words = set(email.split())
    spams = [1.0]
    hams = [1.0]
    for word in words:
        if word in model:
            spams.append(model[word]['spam']/num_spam*total)
            hams.append(model[word]['ham']/num_ham*total)
    prod_spams = np.compat.long(np.prod(spams)*num_spam)
    prod_hams = np.compat.long(np.prod(hams)*num_ham)
    return prod_spams/(prod_spams + prod_hams)

In [14]:
predict_naive_bayes('lottery sale')

0.9638144992048691

In [15]:
predict_naive_bayes('Hi mom how are you')

0.12554358867164467

In [16]:
predict_naive_bayes('Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd')

0.12554358867164467

In [17]:
predict_naive_bayes('meet me at the lobby of the hotel at nine am')

6.964603508395965e-05

In [18]:
predict_naive_bayes('enter the lottery to win three million dollars')

0.9995234218677428

In [19]:
predict_naive_bayes('buy cheap lottery easy money now')

0.999973472265966

In [20]:
predict_naive_bayes('Grokking Machine Learning by Luis Serrano')

0.4197107645488719

In [21]:
predict_naive_bayes('asdfgh')

0.2388268156424581